# Week 9 — Designing & Building Agentic Systems
## Study Notes: LexAgent for Rental Law Reasoning

**Session date:** Week 9 live session
**Instructor note (from transcript):** *"My intent is for you to understand how the code works in general — how you are plugging one function to another, and how the orchestration of the whole thing is working."*

---

### Session Roadmap
| # | Topic |
|---|-------|
| 1 | Problem scenario & motivation |
| 2 | LangChain ecosystem & LangGraph introduction |
| 3 | Tools (5 specialised legal tools) |
| 4 | RAG initialisation & vector stores |
| 5 | ReAct agent workflow (nodes, edges, state) |
| 6 | Running sample queries |
| 7 | Evaluation with DeepEval |
| 8 | Faculty takeaways & design recommendations |

> **Disclaimer (from session):** This agent is for *learning purposes only*. For real legal use, either add a proper disclaimer for end-users, or position it as an assistant for legal experts — never as a replacement for qualified legal counsel.


---
## 1. Problem Scenario

### The challenge
Rental law is:
- **Complex** and jurisdiction-specific (this agent targets **New York City**)
- **Frequently updated** — hard to stay current
- Written in **dense legal language** most people don't understand
- Costly and time-consuming to navigate (lawyers are expensive)

### Proposed solution — LexAgent
An AI-powered **legal intelligence system** that can:

| Capability | Source |
|---|---|
| Look up rental policies & regulations | Web search (Tavily) |
| Analyse prior NYC housing court cases | Vector store (cases PDF) |
| Review & rewrite lease clauses for compliance | LLM rewriting tool |
| Explain legal rights in plain language | Vector store (tenants rights PDF) |
| Evaluate agent performance | DeepEval framework |

> **Instructor quote:** *"Just FYI, this is only for the New York resident area. You have to be very careful when it comes to any type of legal documents — or financial, or healthcare."*


---
## 2. The LangChain Ecosystem & LangGraph

### LangChain family at a glance
```
LangChain        — Core abstractions: LLM wrappers, chains, prompts, tools
LangGraph        — Workflow orchestration as a state graph
LangSmith        — Observability, tracing, evaluation (like a debugger for agents)
LangGraph Dev    — Local dev server / studio for LangGraph
```

### What is LangGraph?
LangGraph models your agentic workflow as a **directed graph**:

```
                    ┌─────────────────────┐
        ┌───────────│      AGENT NODE      │◄──────────┐
        │           │  (LLM + ReAct logic) │           │
        │           └──────────┬──────────┘           │
        │                      │                       │
        │          ┌───────────▼───────────┐           │
        │          │   should_continue()?  │           │
        │          └───────┬───────┬───────┘           │
        │                  │       │                   │
        │             "tools"    "end"                 │
        │                  │       │                   │
        │           ┌──────▼───────┐                   │
        └───────────│  TOOLS NODE  │───────────────────┘
                    │ (invoke tool)│
                    └─────────────┘
```

- **Nodes** = functions (Python) that process state
- **Edges** = connections between nodes
- **Conditional edges** = routing logic (`should_continue` decides: call a tool OR return final answer)
- **State** = shared memory dictionary passed between all nodes

> **Instructor quote:** *"With LangGraph you have more control over your flow — you can have very strict rules for moving between different states, or moving to tool calls. You have very good control over it."*

### LangGraph vs. SmolAgent
| | SmolAgent | LangGraph |
|---|---|---|
| Tool decision | LLM decides autonomously | You define explicit routing |
| Control | Loose — relies on model capability | Strict — enforced workflow |
| Visibility | Black-box reasoning | Full graph visualisation |
| Complexity | Simpler to set up | More code, more control |
| Best for | General-purpose, flexible tasks | Production flows needing guardrails |

> **Instructor quote (Zareh's question):** *"If I wanted to implement it using LangChain only, that would be a better design for this specific case. But if you want to expand it later, I highly recommend LangGraph."*

> **Key insight:** LangGraph is essentially what SmolAgent does internally — but made *explicit and customisable*. You are building the orchestration yourself rather than relying on the model's judgment.


---
## 3. Agent State — The Shared Memory

State is the **central memory** that all nodes read from and write to. It must be defined before the graph.

```python
from typing import Annotated, List, TypedDict
from langchain_core.messages import BaseMessage
import operator

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]  # appends (never replaces)
    step_count: int
```

### Key points about State
1. **Every node receives** the current state as input
2. **Every node returns** a dict with only the fields it updates
3. If you return a field NOT defined in the state → it gets **silently dropped**
4. `Annotated[List[...], operator.add]` means new messages are **appended** to the list, not replaced
5. `step_count` is a simple integer counter — no reducer needed, just overwrite

### Pydantic BaseModel vs TypedDict
| | `TypedDict` | Pydantic `BaseModel` |
|---|---|---|
| Runtime validation | ❌ No — fails silently | ✅ Yes — raises on wrong type |
| Best for | Quick prototyping | Production code |
| Recommendation | Fine for learning | Prefer in real projects |

> **Instructor quote:** *"I myself normally prefer to see something fail during development and make sure I'm not missing anything. That's the reason I normally have the BaseModel."*

### Message types in LangChain
```
SystemMessage   → Instructions/persona for the LLM (sent once, at the top)
HumanMessage    → User query
AIMessage       → LLM response (may include tool_calls)
ToolMessage     → Result from an invoked tool (the "Observation")
```

> **Instructor quote:** *"You have the system message at the very top, then human message and AI message alternating. And we have an optional tool message."*


---
## 4. The Five Specialised Legal Tools

Each tool is decorated with `@tool` from LangChain — this wraps the function with `.invoke()`, `.name`, and `.description` attributes, enabling the agent to discover and call it automatically.

```python
from langchain.tools import tool

@tool
def my_tool(query: str) -> str:
    """Description shown to the LLM so it knows when to use this tool."""
    return "result"
```

> **Instructor quote:** *"If you put the @tool decorator on top, you can invoke it using the .invoke() method. Without it, you just call it like a normal Python function: `my_tool(query)`."*

---

### Tool 1 — `parse_query`
**Purpose:** Validates the query (domain guardrail) and extracts structured metadata.

**What it does:**
1. Checks for tenant/landlord keywords → rejects out-of-scope queries
2. Detects **intent** (one of: eviction, rent increase, security deposit, repairs, etc.)
3. Extracts **jurisdiction** (defaults to New York, US)
4. Identifies **property type** (apartment, house, condo…)
5. Lists **legal issues** (can be multiple: notice requirements, deposit handling, etc.)
6. Flags **missing info**

**Limitation flagged in session:**
> *"I am notorious for making typo mistakes. If I type 'tenant' with an 'a' instead of 'e', it's not going to catch it. I would use an LLM agent to find the intent — not keyword matching."*

This is a design improvement to consider: **replace regex/keyword matching with an LLM-based intent parser** for robustness.

---

### Tool 2 — `policy_search_tool`
**Purpose:** Semantic search over the **tenants_rights.pdf** vector store (36 pages of NYC policy).

```python
@tool
def policy_search_tool(query: str) -> str:
    """Search NYC rental policy documents for relevant regulations."""
    if not policy_vector_store:
        return "Policy vector store not initialised."
    results = policy_vector_store.similarity_search(query, k=2)
    return "\n\n".join([doc.page_content for doc in results])
```

Returns the **2 most semantically similar chunks** from the policy document.

---

### Tool 3 — `cases_search_tool`
**Purpose:** Semantic search over **previous_cases.pdf** (8 historical NYC housing court cases).

Returns the **3 most relevant case chunks** — useful for precedent-based reasoning.

> **Instructor note:** 8 cases → 7 chunks (some cases bleed across chunk boundaries). This is a quality issue.
> **Recommendation:** *"Chunk in a way that every chunk includes one complete case — don't let cases bleed across boundaries."*

---

### Tool 4 — `tavily_web_search_tool`
**Purpose:** Real-time web search for up-to-date NYC housing law news and cases.

Requires a **Tavily API key** (free tier: 1,000 searches/month at tavily.com).

```python
@tool
def tavily_web_search_tool(query: str) -> str:
    """Search the web for recent NYC housing law developments."""
    tavily = TavilySearchResults(max_results=5, search_depth="advanced",
                                  include_answer=True, include_images=False)
    results = tavily.invoke(query)
    # formats results into a single string with URLs
    ...
```

---

### Tool 5 — `rewrite_legal_clause`
**Purpose:** Rewrites a lease clause to be NYC-compliant while favouring the landlord perspective.

**Special input — uses a Pydantic model:**
```python
class RewriteClauseInput(BaseModel):
    original_clause: str
    parse_claim: str
    policy_excerpt: str
    case_log: str
```

The LLM prompt instructs: *"Rewrite the original clause so it fully complies while remaining as landlord-favourable as legally permitted. Do not add legal analysis or citations."*

> **Instructor quote:** *"The input for that one is always going to be a dictionary including the original clause, parse claim, policy excerpt, and case log."*

---

### Tool Map — Dynamic Discovery
Rather than hard-coding tool names, the agent builds a map dynamically:

```python
tool_list = [parse_query, policy_search_tool, cases_search_tool,
             tavily_web_search_tool, rewrite_legal_clause]

# Maps name → tool object for invocation
tool_map = {t.name: t for t in tool_list}

# Builds description string for the prompt (so LLM knows what tools exist)
TOOLS_DESC = render_text_description(tool_list)
```

> **Instructor quote:** *"What we are doing here is that we wanted to get the names and descriptions of the tools from this structure — because we don't know how many tools we have, and we want to be able to add or remove tools."*


---
## 5. RAG System — Vector Store Initialisation

### The 4-step RAG pipeline
```
1. Load     → PyPDFLoader reads PDF pages
2. Chunk    → RecursiveCharacterTextSplitter splits into overlapping chunks
3. Embed    → OpenAI text-embedding-3-small converts text → vectors
4. Store    → ChromaDB persists vectors for similarity search
```

```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

def initialize_rag_system(pdf_path: str, rag_type: str):
    global policy_vector_store, case_vector_store

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = splitter.split_documents(documents)

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    if rag_type == 'policies':
        policy_vector_store = Chroma.from_documents(splits, embeddings,
                                                     collection_name="policies")
    elif rag_type == 'cases':
        case_vector_store = Chroma.from_documents(splits, embeddings,
                                                   collection_name="cases")

# Initialise both
initialize_rag_system('tenants_rights.pdf', 'policies')   # → 126 chunks
initialize_rag_system('previous_cases.pdf', 'cases')      # → 7 chunks (8 cases)
```

### Global variables pattern
```python
global policy_vector_store, case_vector_store  # declared inside function
```
> **Instructor quote:** *"Any variables you define outside a function can be called from any place in your notebook. If you define something inside a function and you don't return it, you won't be able to access it later — it's a local variable. The `global` keyword lets you modify the outer scope variable from inside the function."*

### PDF quality warning
The PyPDFLoader produces text with excessive whitespace between words. This wastes tokens and can confuse the model.
**Fix:** Clean text after loading:
```python
for doc in documents:
    doc.page_content = re.sub(r'\s+', ' ', doc.page_content).strip()
```

### Two vector stores vs. one
The notebook uses two separate Chroma instances. A better practice:
> *"Use one single vector store with two different collections — like having one database with separate tables."*


---
## 6. ReAct Agent Workflow

### The ReAct pattern
**ReAct = Reasoning + Acting**

The LLM alternates between:
```
Thought:  [reasoning about what to do next]
Action:   [tool name]
Action Input: [tool arguments]
--- (tool runs, returns observation) ---
Observation: [tool output]
Thought:  [reasoning about the observation]
... repeat ...
Final Answer: [complete response to the user]
```

### The ReAct Prompt Template
```python
REACT_TEMPLATE = """
You are a legal reasoning assistant using the ReAct framework.
Your job is to answer queries about New York tenant law.

Tools available:
{tools}

Format:
Thought: [your reasoning]
Action: [tool name]
Action Input: [tool input]
Observation: [tool result — provided automatically]
... (repeat as needed) ...
Thought: I now have the final answer.
Final Answer: [your answer]

Question: {question}
{observation}
"""
```

> **Instructor quote:** *"This is the same things that we discussed several times — thought, action, action input, observation as needed."*

---

### The Agent Node (`agent_node`)

The brain of the system. Called on every iteration of the loop.

```python
def agent_node(state: AgentState):
    messages = state["messages"]
    step_count = state.get("step_count", 0)

    # 1. Guard: prevent infinite loops
    if step_count >= 6:
        return {"messages": [], "step_count": step_count}

    # 2. Build context
    question = messages[0].content
    observation = ""
    if isinstance(messages[-1], ToolMessage):
        observation = f"Observation:\n{messages[-1].content}"

    # 3. Call LLM with formatted ReAct prompt
    formatted_prompt = prompt.format(tools=TOOLS_DESC, question=question,
                                      observation=observation)
    response = llm.invoke([
        {"role": "system", "content": "Follow ReAct strictly."},
        {"role": "user", "content": formatted_prompt},
    ])
    llm_output = response.content

    # 4. Parse LLM output — either AgentAction or AgentFinish
    parser = ReActSingleInputOutputParser()
    parsed = parser.parse(llm_output)

    if isinstance(parsed, AgentFinish):
        # Done — return final answer
        return {
            "messages": [AIMessage(content=parsed.return_values["output"])],
            "step_count": step_count + 1,
        }

    if isinstance(parsed, AgentAction):
        # Need to call a tool — prepare AIMessage with tool_calls
        tool_name = parsed.tool
        # Coerce args based on tool type...
        return {
            "messages": [AIMessage(content="", tool_calls=[{
                "name": tool_name, "args": args, "id": "tc_" + tool_name
            }])],
            "step_count": step_count + 1,
        }
```

**Key design point:** The agent doesn't *run* the tool. It *requests* the tool by returning an `AIMessage` with `tool_calls`. The `tools_node` actually executes it.

> **Instructor quote:** *"This agent decides: do I need to call a tool, or do I already have the answer?"*

---

### The Tools Node (`tools_node`)

Executes whatever tool the agent requested.

```python
def tools_node(state: AgentState):
    last = state["messages"][-1]
    tool_call = last.tool_calls[0]

    tool_name = tool_call["name"]
    args = tool_call["args"]

    tool = tool_map[tool_name]
    observation = tool.invoke(args)

    return {
        "messages": [ToolMessage(
            content=str(observation),
            tool_call_id=tool_call["id"],
            name=tool_name,
        )],
        "step_count": state["step_count"],  # step_count unchanged here
    }
```

> **Instructor quote:** *"The step count is only incrementing from the agent. The tool node doesn't add to it."*

---

### The Routing Function (`should_continue`)

A **conditional edge** function — determines where to go after the agent node.

```python
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    messages = state["messages"]
    if not messages:
        return END
    last = messages[-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END
```

---

### Assembling the Graph

```python
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tools_node)

# Set entry point
workflow.set_entry_point("agent")

# Conditional edge: after agent → tools OR end
workflow.add_conditional_edges("agent", should_continue,
                                {"tools": "tools", END: END})

# Fixed edge: after tools → always back to agent
workflow.add_edge("tools", "agent")

app = workflow.compile()
```

### Visualising the graph
```python
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())
```
This renders the Mermaid diagram shown at the top of the notebook.


---
## 7. Running Queries — Three Demonstration Cases

### `run_legal_agent` — The Entry Point

```python
def run_legal_agent(query: str):
    for step in app.stream(
        {"messages": [HumanMessage(content=query)], "step_count": 0},
        stream_mode="values",
    ):
        messages = step.get("messages", [])
        if not messages:
            continue
        last = messages[-1]
        if isinstance(last, AIMessage) and not last.tool_calls:
            print(last.content)
            break
```

`app.stream()` yields the **full state** after every node execution — useful for observing each step live.

---

### Run 1 — Security Deposit Legality

**Query:** *"I'm planning to rent an apartment in NYC under a 3-year lease. The landlord is demanding 12 months' rent as a security deposit. Is this legal?"*

**What the agent does (from session demo):**
1. `parse_query` → intent: Security Deposit Dispute
2. `policy_search_tool` → policy search didn't return specific NYC deposit limit
3. `cases_search_tool` → tried to find in case law
4. `tavily_web_search_tool` → web search for NYC 12-month deposit
5. **Final Answer:** NYC law generally limits security deposits to **1 month's rent** for most residential leases — 12 months is illegal.

---

### Run 2 — Non-Payment & Unauthorised Subletting

**Query:** Landlord wants eviction for unpaid rent AND unauthorised subletting.

**What the agent does:**
- Calls multiple tools (parse → policy → cases → web)
- Synthesises the ReAct Thought-Action-Observation loop across multiple steps
- Provides step-by-step legal action plan for eviction proceedings

> *"He's going through the thoughts — thought process — and it's taking action every time, collecting all the answers, and finalising."*

---

### Run 3 — Clause Rewriting

**Query:** Rewrite a specific lease clause for NYC compliance while maximising landlord protections.

**What the agent does:**
1. Parses the query → detects rewrite intent
2. Searches policies for compliance requirements
3. Calls `rewrite_legal_clause` with structured input:

```python
{
    "original_clause": "...",
    "parse_claim": "...",    # from parse_query
    "policy_excerpt": "...", # from policy_search
    "case_log": "...",       # from cases_search
}
```

> **Instructor quote:** *"The input for rewrite_legal_clause is always going to be a dictionary. This is different from the other tools where the input is just the query string."*


---
## 8. Evaluation with DeepEval

### Why evaluate?
> *"The agent perfectly works. But how do I know this is working RIGHT? How do I know this is high-quality?"*

DeepEval is an open-source LLM evaluation framework. It uses a **judge LLM** to score agent outputs against defined metrics.

### Installation
```bash
pip install deepeval
```

### The three metrics used

| Metric | What it measures | Threshold |
|---|---|---|
| `ToolCorrectnessMetric` | Did the agent call the expected tools? | 0.7 (70%) |
| `TaskCompletionMetric` | Did the agent complete the task? | 0.7 |
| `AnswerRelevancyMetric` | Is the answer relevant to the query? | 0.7 |

---

### Setting up evaluation

**Step 1: Utility functions** (extract what the agent actually did)
```python
def extract_tool_calls_from_state(state):
    """Returns list of (tool_name, tool_input) pairs from agent messages."""
    ...

def convert_to_tool_calls(tool_info):
    """Converts to DeepEval ToolCall objects."""
    return [ToolCall(name=t["name"], input_parameters=t["args"]) for t in tool_info]

def get_final_output(state):
    """Gets the last AIMessage content."""
    ...

def create_expected_tools():
    """Hard-coded expected tool sequence for a specific query."""
    return [
        ToolCall(name="parse_query", ...),
        ToolCall(name="policy_search_tool", ...),
        ToolCall(name="cases_search_tool", ...),
        ToolCall(name="tavily_web_search_tool", ...),
    ]
```

**Step 2: Judge LLM**
```python
from deepeval.models import GPTModel
judge_llm = GPTModel(model="gpt-4o-mini")
```

**Step 3: Test functions**
```python
def test_tool_correctness(query, state, actual_tools, expected_tools, final_output):
    test_case = LLMTestCase(
        input=query,
        actual_output=final_output,
        tools_called=actual_tools,
        expected_tools=expected_tools,
    )
    metric = ToolCorrectnessMetric(threshold=0.7, model=judge_llm)
    metric.measure(test_case)
    print(f"Score: {metric.score}, Pass: {metric.is_successful()}")
    print(f"Reason: {metric.reason}")
```

**Step 4: Run evaluation**
```python
run_evaluation(test_query)
# Output:
# Tool Correctness: score=1.0  PASSED ✓
# Task Completion:  score=1.0  PASSED ✓
# Answer Relevancy: score=1.0  PASSED ✓
# Overall: 3/3 PASSED
```

**Deliberate failure demo** — swap in wrong expected tools:
```python
# Force-set wrong expected tools → ToolCorrectnessMetric drops to 0.5 → FAIL
```

---

### DeepEval — Important Caveats

> **Instructor quote:** *"The tool calling is manually crafted. We are hard-coding and saying: for that example, I'm expecting to see those tool calls happen. This is not something that automatically changes."*

> **Dino's insight:** *"Tool correctness could be right — but it's not a matter of selecting the wrong tools. It's evaluating the reasoning steps, like, was the path efficient? I'm wondering if there are benchmarks for that."*

> **Instructor response:** DeepEval has **custom metrics** — you can define expected reasoning steps and compare. But you need to hard-code the expected behaviour for each specific test case.

### Beyond the 3 metrics
DeepEval has many more built-in metrics:
- Bias & Toxicity detection
- RAG-specific metrics (faithfulness, context precision/recall)
- MCP safety metrics
- Conversation metrics
- Custom G-Eval metrics


---
## 9. Faculty Takeaways & Design Recommendations

These are direct recommendations from the instructor at the end of the session.

---

### Takeaway 1 — Architecture: Use a Workflow, Not a ReAct Agent

> *"If I were to design this, I would use it as a workflow. I would have one agent for parsing the intent, a router that says: if intent is X → policy + web search, if intent is rewrite → separate workflow. You don't need much of a ReAct for this case."*

**Why?** ReAct is powerful but overkill when the flow is predictable:
- Parse → Retrieve → Answer is a **linear pipeline**
- A LangChain chain with explicit routing is simpler, faster, cheaper

**Design proposal:**
```
User Query
    │
    ▼
[Intent Router]  ← LLM classifies intent
    │
    ├── LEGAL QUESTION → [parse_query] → [policy_search] → [case_search] → [web_search] → [LLM answer]
    │
    └── REWRITE REQUEST → [parse_query] → [policy_search] → [case_search] → [rewrite_legal_clause]
```

---

### Takeaway 2 — Replace Keyword Matching with LLM in `parse_query`

Current implementation uses `if "tenant" in query.lower()` — brittle and typo-sensitive.

**Recommendation:** Use an LLM call to extract:
- Intent
- Jurisdiction
- Property type
- Legal issues

```python
# Example using structured output
from langchain_core.output_parsers import JsonOutputParser

parse_chain = (
    ChatPromptTemplate.from_template(
        "Extract these fields from the query: {fields}\nQuery: {query}\nReturn JSON."
    )
    | llm
    | JsonOutputParser()
)
```

---

### Takeaway 3 — Fix Chunking Strategy for Case Documents

> *"If you recall, that was 7 chunks for 8 cases — some cases are missing or bleeding from the previous one. This is bad practice."*

**Current problem:**
- Fixed-size chunking splits individual cases across chunk boundaries
- Retrieval may return partial cases → incomplete context

**Recommendation:** Use **document-aware chunking**:
```python
# Each case is a separate document — don't let them bleed
# Option 1: Split on case boundaries ("Case 1:", "Case 2:", etc.)
# Option 2: Load each case as a separate Document before embedding
```

---

### Takeaway 4 — Clean PDF Text After Loading

> *"The PDF loader output is kind of gibberish and not good — there are two line breaks between every word. I highly recommend fixing it."*

```python
import re

# After loader.load(), clean the text
for doc in documents:
    doc.page_content = re.sub(r'\s+', ' ', doc.page_content).strip()
```

---

### Takeaway 5 — One Vector Store, Two Collections (Not Two Stores)

> *"Use one single vector store but have two different collections — similar to having one database schema with separate tables. The management is far easier."*

```python
# Instead of two Chroma instances:
policy_store = Chroma(collection_name="policies", ...)
case_store    = Chroma(collection_name="cases", ...)

# Use one client, two collections:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")
policy_collection = Chroma(client=client, collection_name="policies", ...)
case_collection   = Chroma(client=client, collection_name="cases", ...)
```

---

### Takeaway 6 — Simplify Agent Code with LangChain Chains

> *"You could have used a prompt template, pipe it to your LLM, pipe it to your output parser — and reduce the agent node to a single line. Much cleaner writing patterns, easier to maintain."*

```python
# Current pattern (verbose):
formatted_prompt = prompt.format(tools=TOOLS_DESC, question=question, ...)
response = llm.invoke([{"role": "system", ...}, {"role": "user", "content": formatted_prompt}])
llm_output = response.content
parsed = ReActSingleInputOutputParser().parse(llm_output)

# LangChain chain pattern (concise):
chain = prompt | llm | ReActSingleInputOutputParser()
parsed = chain.invoke({"tools": TOOLS_DESC, "question": question, "observation": obs})
```

---

### Takeaway 7 — Use Pydantic BaseModel for State (Not TypedDict)

> *"I highly recommend using the BaseModel from Pydantic. It's gonna make trouble for you — but in a good way, not a bad way. It fails fast during development."*

---

### Takeaway 8 — Connecting Graphs Across Runtimes (Advanced)

**From class discussion (Dino + Zareh):**

Options for connecting two LangGraph agents in different runtimes:
1. **Nested/subgraphs** — if same codebase, compile one graph inside another
2. **REST API** — expose the graph as a FastAPI endpoint; other agents call it via HTTP ✅ *Faculty recommendation*
3. **MCP server** — valid but overengineered for agent-to-agent; better for resource access
4. **A2A protocol** — appropriate when agents are in entirely different frameworks

> *Faculty:* "REST API is the practice that everyone is using. Those two runtimes are completely isolated but communicate through the REST API."
> *Zareh:* "We've exposed LangGraph as MCP tools and as REST APIs — it works for cross-runtime graph calls."
> *Dino:* "I'd use A2A if we're on different frameworks, or MCP if it's a resource, but for agent-to-agent within LangGraph, REST API makes sense."


---
## 10. Annotated Code Samples

These are minimal, standalone examples of the key patterns from this session.


In [ ]:
# ── Pattern 1: Define State with TypedDict ────────────────────────────────
from typing import Annotated, List, TypedDict
from langchain_core.messages import BaseMessage
import operator

class AgentState(TypedDict):
    # operator.add reducer means new messages are APPENDED, not replaced
    messages: Annotated[List[BaseMessage], operator.add]
    step_count: int

# Usage: agent nodes read state["messages"] and state["step_count"]
print("AgentState keys:", AgentState.__annotations__.keys())


In [ ]:
# ── Pattern 2: Build a LangGraph in 5 lines ───────────────────────────────
from langgraph.graph import StateGraph, END
from typing import Literal

# Define routing function
def router(state) -> Literal["tools", "__end__"]:
    last = state["messages"][-1] if state["messages"] else None
    if last and hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

# Build graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", lambda s: s)     # replace with agent_node
workflow.add_node("tools", lambda s: s)     # replace with tools_node
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", router, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")         # tools always return to agent
app = workflow.compile()

print("Graph compiled. Nodes:", list(workflow.nodes.keys()))


In [ ]:
# ── Pattern 3: LangChain chain (simplified agent_node) ────────────────────
# This is the faculty-recommended refactor of agent_node

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Define components
prompt = PromptTemplate.from_template(
    """You are a legal reasoning assistant using ReAct.

Available tools:
{tools}

Respond in this format:
Thought: ...
Action: [tool name]
Action Input: [tool input]

OR if you have the final answer:
Final Answer: [your answer]

Question: {question}
{observation}"""
)

# Chain: prompt | llm | parser
# chain = prompt | llm | ReActSingleInputOutputParser()

# Usage in agent_node:
# parsed = chain.invoke({"tools": TOOLS_DESC, "question": q, "observation": obs})
print("Chain pattern ready (needs live LLM to invoke)")


In [ ]:
# ── Pattern 4: DeepEval quick test ────────────────────────────────────────
# Minimal example of how to use DeepEval metrics

# from deepeval.test_case import LLMTestCase, ToolCall
# from deepeval.metrics import ToolCorrectnessMetric
# from deepeval.models import GPTModel

# judge = GPTModel(model="gpt-4o-mini")
# metric = ToolCorrectnessMetric(threshold=0.7, model=judge)

# test_case = LLMTestCase(
#     input="Is a 12-month security deposit legal in NYC?",
#     actual_output="No, NYC limits deposits to 1 month's rent.",
#     tools_called=[
#         ToolCall(name="parse_query", input_parameters={"query": "..."}),
#         ToolCall(name="policy_search_tool", input_parameters={"query": "..."}),
#     ],
#     expected_tools=[
#         ToolCall(name="parse_query", ...),
#         ToolCall(name="policy_search_tool", ...),
#     ],
# )

# metric.measure(test_case)
# print(f"Score: {metric.score}, Pass: {metric.is_successful()}")
# print(f"Reason: {metric.reason}")
print("DeepEval example (commented out — needs API keys to run)")


---
## 11. Questions & Answers from the Live Session

These are actual questions and answers from the class discussion.

---

**Q (Ananda):** How is LangGraph different from SmolAgent?

**A:** SmolAgent is a *single-point agent* — everything (reasoning, tool selection, output) happens inside the model's "brain." LangGraph *externalises* those steps. You define explicit nodes for each stage and explicit edges for routing. The result is more control and visibility, at the cost of more code.

---

**Q (Ananda):** Can the state be persisted somewhere like a database?

**A:** Yes. LangGraph supports checkpointers (e.g., `SqliteSaver`, `PostgresSaver`) that persist the state after each step. This enables resumable workflows and conversation memory across sessions.

---

**Q (Zareh):** Do you really need LangGraph for this use case?

**A:** No — and the faculty agreed! For this specific notebook, LangChain with explicit chains would be *simpler and better*. LangGraph is most valuable when you need **cyclic workflows** (agent-tool loops), **branching logic**, or **visualisation**. Linear pipelines work fine with plain chains.

---

**Q (Dino):** Can you call a LangGraph from another LangGraph in a different runtime?

**A:** Yes — several approaches:
1. **Subgraphs** (same runtime): compile one graph inside another
2. **REST API** (different runtimes): expose as FastAPI endpoint ← *recommended*
3. **MCP server** (cross-framework resources)
4. **A2A protocol** (different frameworks entirely)

---

**Q (Dino):** Can DeepEval evaluate the *quality of reasoning*, not just tool correctness?

**A:** Yes. DeepEval has `GEval` (custom G-Eval) where you define your own criteria. But you need to specify expected behaviour hard-coded per test case. For general reasoning quality, you'd need to define what "good reasoning" looks like for your domain.

---

**Q (Zareh):** Do you use DeepEval at work?

**A:** The faculty uses it, but for production systems the metrics were too SME-specific. For customer service chatbots, work-order systems, etc., the quality criteria are unique enough that a custom LLM judge was built instead.

---

**Q (Zareh):** Is there a more comprehensive evaluation framework?

**A:** LangSmith has evaluation built in (Dino also recommended it for observability). DeepEval is more comprehensive for standalone evaluation. Both are valid — LangSmith is easiest if you're already in the LangChain ecosystem.

---

**Q (Dino about best naming for nodes/functions):**

**A:** Faculty recommends suffix convention: `_node` for node functions, `_edge` for routing functions. Keeps the graph definition readable.


---
## 12. Key Terms Glossary

| Term | Definition |
|---|---|
| **ReAct** | Reasoning + Acting — alternates Thought / Action / Observation loops |
| **LangGraph** | LangChain extension for graph-based workflow orchestration |
| **StateGraph** | LangGraph class for building stateful agent graphs |
| **AgentState** | Shared memory dictionary passed between all graph nodes |
| **`operator.add`** | Reducer that *appends* to a list instead of replacing it |
| **Node** | A Python function that processes state in the graph |
| **Edge** | Connection between nodes; can be fixed or conditional |
| **Conditional Edge** | Routing function that picks the next node based on state |
| **`END`** | LangGraph sentinel — terminates the graph execution |
| **AgentAction** | ReAct parser output when LLM wants to call a tool |
| **AgentFinish** | ReAct parser output when LLM has the final answer |
| **ToolMessage** | LangChain message type carrying a tool's output (Observation) |
| **RAG** | Retrieval-Augmented Generation — grounds LLM with retrieved documents |
| **ChromaDB** | Embedded vector database used for semantic search |
| **Tavily** | Search API for real-time web lookup (1,000 free queries/month) |
| **DeepEval** | LLM evaluation framework with built-in metrics |
| **Judge LLM** | An LLM used to evaluate the quality of another LLM's output |
| **ToolCorrectnessMetric** | DeepEval metric: were the right tools called? |
| **TaskCompletionMetric** | DeepEval metric: did the agent complete the task? |
| **AnswerRelevancyMetric** | DeepEval metric: is the answer relevant to the query? |
| **`render_text_description`** | LangChain utility that formats tool list → string for prompts |


---
## 13. Further Reading & Next Steps

### Official documentation
- LangGraph: https://langchain-ai.github.io/langgraph/
- DeepEval: https://docs.confident-ai.com/
- Tavily: https://docs.tavily.com/

### Concepts to explore next
1. **LangGraph checkpointers** — persist state to SQLite/Postgres for memory across sessions
2. **LangSmith tracing** — full observability for debugging agent runs
3. **LangGraph Studio** — visual debugger for graphs
4. **Subgraphs** — compose multiple graphs together
5. **Multi-agent patterns** — supervisor, worker, hierarchical graphs
6. **Structured outputs** — use `llm.with_structured_output(schema)` instead of ReAct parser
7. **LangGraph Cloud** — deploy graphs as managed APIs

### Exercises (from session spirit)
1. Replace keyword matching in `parse_query` with an LLM-based extractor
2. Fix the PDF chunking to keep each case as a single document
3. Clean PDF text after loading (remove excess whitespace)
4. Refactor `agent_node` to use a LangChain chain (`prompt | llm | parser`)
5. Add LangSmith tracing to observe the full execution trace
6. Add a `DISCLAMER` node that appends a legal disclaimer to every Final Answer
7. Extend to a second jurisdiction (e.g., California) with a router
